# 03 - Run Hybrid Models

Trains hybrid quantum-classical students (NoKD and KD).

**Recommended env:** `conda activate hybridqml311` (Python 3.11)

Run in two stages:
1. **Smoke test** (`n_splits=2, epochs=5`) — verify everything works (~2-3 min)
2. **Full run** (`n_splits=5, epochs=60`) — produce paper results (~15-30 min)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.utils.seed import set_seed
from src.trainers.train_teacher import train_teacher_cv
from src.trainers.train_student import run_student_cv

set_seed(42)
CSV = os.path.abspath('../data/raw/wdbc.csv')
assert os.path.exists(CSV), f'Data file not found: {CSV}'
print('CSV ready:', CSV)

CSV ready: /Users/isaac/clawd/research/HybridQMedKD/data/raw/wdbc.csv


## Stage 1 — Smoke Test (run this first)

In [7]:
# ---- Smoke test parameters ----
SMOKE = False   # set to False for full run

N_SPLITS = 2  if SMOKE else 5
EPOCHS   = 5  if SMOKE else 60
BATCH    = 8  if SMOKE else 16
N_Q_LAYERS = 1
PCA_DIM    = 4
N_QUBITS   = 4
print(f'Mode: {"SMOKE" if SMOKE else "FULL"}  | splits={N_SPLITS}  epochs={EPOCHS}  batch={BATCH}')

Mode: FULL  | splits=5  epochs=60  batch=16


## Step 1 — Train Teacher

In [8]:
teacher = train_teacher_cv(
    CSV, pca_dim=PCA_DIM, n_splits=N_SPLITS, seed=42,
    epochs=40 if SMOKE else 80
)
print('Teacher done.')

[Teacher] Fold 1/5
  AUC=0.9961  F1=0.9451  MCC=0.9125  Time=0.3s
[Teacher] Fold 2/5
  AUC=0.9941  F1=0.9136  MCC=0.8702  Time=0.3s
[Teacher] Fold 3/5
  AUC=0.9845  F1=0.9383  MCC=0.9058  Time=0.3s
[Teacher] Fold 4/5
  AUC=0.9980  F1=0.9438  MCC=0.9119  Time=0.3s
[Teacher] Fold 5/5
  AUC=0.9987  F1=0.9630  MCC=0.9439  Time=0.3s

[Teacher] Mean AUC=0.9943  Mean F1=0.9407
Teacher done.


## Step 2 — Student Hybrid NoKD

In [9]:
metrics_nokd = run_student_cv(
    CSV,
    teacher_fold_outputs=None,
    model_type='hybrid',
    use_kd=False,
    quantum_position='middle',
    pca_dim=PCA_DIM,
    n_qubits=N_QUBITS,
    n_q_layers=N_Q_LAYERS,
    n_splits=N_SPLITS,
    seed=42,
    epochs=EPOCHS,
    lr=1e-3,
    batch_size=BATCH,
    exp_name='student_hybrid_nokd'
)
print('Hybrid NoKD done.')

[student_hybrid_nokd] Fold 1/5
  epoch   1/60  loss=0.6357  elapsed=5.0s
  epoch   5/60  loss=0.5604  elapsed=24.2s
  epoch  10/60  loss=0.4374  elapsed=48.4s
  epoch  15/60  loss=0.3251  elapsed=72.5s
  epoch  20/60  loss=0.2346  elapsed=96.5s
  epoch  25/60  loss=0.1811  elapsed=120.6s
  epoch  30/60  loss=0.1477  elapsed=144.5s
  epoch  35/60  loss=0.1278  elapsed=168.6s
  epoch  40/60  loss=0.1164  elapsed=192.9s
  epoch  45/60  loss=0.1067  elapsed=217.1s
  epoch  50/60  loss=0.0989  elapsed=241.2s
  epoch  55/60  loss=0.0951  elapsed=265.4s
  epoch  60/60  loss=0.0896  elapsed=289.7s
  -> AUC=0.9967  F1=0.9556  MCC=0.9292  Train=289.7s
[student_hybrid_nokd] Fold 2/5
  epoch   1/60  loss=0.6610  elapsed=4.9s
  epoch   5/60  loss=0.6147  elapsed=24.4s
  epoch  10/60  loss=0.4417  elapsed=48.8s
  epoch  15/60  loss=0.3026  elapsed=73.3s
  epoch  20/60  loss=0.2050  elapsed=97.8s
  epoch  25/60  loss=0.1457  elapsed=122.1s
  epoch  30/60  loss=0.1134  elapsed=146.3s
  epoch  35/60  l

## Step 3 — Student Hybrid KD

In [10]:
metrics_kd = run_student_cv(
    CSV,
    teacher_fold_outputs=teacher,
    model_type='hybrid',
    use_kd=True,
    alpha=0.5,
    T=2.0,
    quantum_position='middle',
    pca_dim=PCA_DIM,
    n_qubits=N_QUBITS,
    n_q_layers=N_Q_LAYERS,
    n_splits=N_SPLITS,
    seed=42,
    epochs=EPOCHS,
    lr=1e-3,
    batch_size=BATCH,
    exp_name='student_hybrid_kd'
)
print('Hybrid KD done.')

[student_hybrid_kd] Fold 1/5
  epoch   1/60  loss=1.4955  elapsed=4.8s
  epoch   5/60  loss=1.2554  elapsed=24.1s
  epoch  10/60  loss=0.9726  elapsed=48.2s
  epoch  15/60  loss=0.7473  elapsed=72.2s
  epoch  20/60  loss=0.5929  elapsed=93.5s
  epoch  25/60  loss=0.5229  elapsed=111.4s
  epoch  30/60  loss=0.4719  elapsed=128.8s
  epoch  35/60  loss=0.4332  elapsed=146.1s
  epoch  40/60  loss=0.3879  elapsed=163.3s
  epoch  45/60  loss=0.3595  elapsed=180.5s
  epoch  50/60  loss=0.3352  elapsed=197.9s
  epoch  55/60  loss=0.3026  elapsed=216.3s
  epoch  60/60  loss=0.2868  elapsed=233.8s
  -> AUC=0.9951  F1=0.9231  MCC=0.8759  Train=233.8s
[student_hybrid_kd] Fold 2/5
  epoch   1/60  loss=1.5111  elapsed=4.8s
  epoch   5/60  loss=1.2533  elapsed=23.9s
  epoch  10/60  loss=0.9021  elapsed=48.1s
  epoch  15/60  loss=0.6086  elapsed=72.0s
  epoch  20/60  loss=0.4125  elapsed=96.2s
  epoch  25/60  loss=0.2922  elapsed=119.9s
  epoch  30/60  loss=0.2157  elapsed=143.7s
  epoch  35/60  loss=

## Results Summary

In [11]:
import numpy as np

def print_summary(name, metrics):
    print(f'--- {name} ---')
    for k in ['auc', 'f1', 'mcc', 'acc']:
        vals = [m[k] for m in metrics]
        print(f'  {k}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}')
    t = [m['train_time'] for m in metrics]
    print(f'  train_time: {np.mean(t):.1f}s/fold')

print_summary('Hybrid-NoKD', metrics_nokd)
print_summary('Hybrid-KD',   metrics_kd)

--- Hybrid-NoKD ---
  auc: 0.9896 +/- 0.0105
  f1: 0.9442 +/- 0.0220
  mcc: 0.9129 +/- 0.0343
  acc: 0.9579 +/- 0.0170
  train_time: 330.5s/fold
--- Hybrid-KD ---
  auc: 0.9931 +/- 0.0054
  f1: 0.9395 +/- 0.0152
  mcc: 0.9057 +/- 0.0249
  acc: 0.9543 +/- 0.0128
  train_time: 276.5s/fold
